# Q1c — Kafka listener: proof of decoupling, resume and at-least-once

Live view of `wiki.recentchange.raw` from **before** the producer starts until **after**
it exits, plus the Q1c decoupling experiment (stop the consumer ~2 minutes while the
producer keeps writing, then restart it).

**Run order** — the notebook must be listening *before* the producer runs:

| step | where | action |
|---|---|---|
| 1 | this notebook | run **Step 1** (`tail.start()`) — listener is up, 0 records yet |
| 2 | JupyterLab *Terminal* | `python3 q1_wikimedia_producer.py` (12 min) |
| 3 | this notebook | **Step 2** — watch the live panel fill up |
| 4 | this notebook | **Step 3** — stop the consumer for `STOP_SECONDS` (120 s); the producer keeps running |
| 5 | this notebook | **Step 4** — restart; it resumes from the committed offset |
| 6 | this notebook | **Step 5** — verification table (lost / duplicated / resumed) |
| 7 | this notebook, after the producer exits | **Step 6** — the topic goes idle, nothing is lost behind the listener |

**What is being demonstrated**

* the consumer group's **committed offsets** are what makes the restart lossless —
  `enable_auto_commit=False` commits only after a batch is processed;
* the topic is a **buffer**: while the consumer is down the lag grows, and it drains
  afterwards, so producer and consumer are decoupled in time;
* delivery is **at-least-once**: a crash between processing and commit can replay
  records, so the `(wiki, id)` key is used to make the processing duplicate-tolerant.

The panel refreshes inside the cell; **Ctrl-C stops the view only** — the background
listener keeps consuming and committing.


In [1]:
import os, sys, json, time, threading, datetime as _dt
from collections import deque, Counter

# kafka-python lives in the jupyter-work volume: survives container re-creation
for _p in ("/opt/notebooks/.pylibs", os.path.join(os.getcwd(), ".pylibs")):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

from kafka import KafkaConsumer
from kafka.structs import TopicPartition

BOOTSTRAP    = os.environ.get("KAFKA_BOOTSTRAP", "localhost:29092")
TOPIC        = "wiki.recentchange.raw"
GROUP_ID     = "wiki-monitor-q1c"   # fixed ON PURPOSE: committed offsets are the evidence
PARTITIONS   = 6                    # must match the topic (kafka-topics.sh --describe)
STOP_SECONDS = 120                  # Q1c: consumer down for ~2 minutes
VIEW_SECONDS = 120                  # how long each live panel refreshes
TABLE_ROWS   = 12                   # rows in the live message table
MAX_RECORDS  = 500                  # records per poll() batch

print(f"broker={BOOTSTRAP}  topic={TOPIC}  group={GROUP_ID}")
print(f"partitions={PARTITIONS}  Q1c downtime window={STOP_SECONDS}s")


broker=10.8.0.6:29092  topic=wiki.recentchange.raw  group=wiki-monitor-q1c
partitions=6  Q1c downtime window=120s


In [2]:
def _fmt_epoch(epoch):
    """Unix seconds -> local HH:MM:SS (topic timestamps are epoch seconds)."""
    try:
        return _dt.datetime.fromtimestamp(int(epoch)).strftime("%H:%M:%S")
    except (TypeError, ValueError, OSError):
        return "-"


class KafkaTail:
    """Consumes TOPIC on a background thread and commits offsets after every batch.

    enable_auto_commit=False is what makes the Q1c test meaningful: the offset that
    survives a stop is exactly the last record this listener *processed*, not a
    record that was still sitting in a poll buffer or an interval that had not fired.
    """

    def __init__(self, bootstrap, topic, group_id, table_rows=12, max_records=500):
        self.bootstrap = bootstrap
        self.topic = topic
        self.group_id = group_id
        self.table_rows = table_rows
        self.max_records = max_records

        self.lock = threading.Lock()
        self._stop = threading.Event()
        self._thread = None
        self.consumer = None

        self.session = 0
        self.session_first = {}      # partition -> first offset seen in THIS session
        self.stop_mark = None        # evidence captured by stop()
        self.last_error = None

        self.count = 0
        self.dup_offsets = 0
        self.skipped = 0             # offsets missing *inside* one session
        self.bad_json = 0
        self.dup_ids = 0
        self.seen_ids = set()
        self.per_part = {}           # partition -> {first, last, count}
        self.committed = {}          # partition -> last committed offset
        self.end = {}                # partition -> end offset (broker)
        self.recent = deque(maxlen=400)
        self.samples = deque(maxlen=2000)
        self.first_ts = None
        self.last_ts = None
        self.started_at = None
        self.stopped_at = None

    # ---------------------------------------------------------------- lifecycle
    def start(self):
        if self._thread and self._thread.is_alive():
            return f"listener already running (session {self.session})"
        self._stop.clear()
        self.session += 1
        self.session_first = {}
        self.consumer = KafkaConsumer(
            self.topic,
            bootstrap_servers=self.bootstrap.split(","),
            group_id=self.group_id,
            enable_auto_commit=False,      # manual commit -> at-least-once, no loss on stop
            auto_offset_reset="earliest",  # first ever run starts at offset 0
            max_poll_records=self.max_records,
            client_id=f"q1c-listener-s{self.session}",
            session_timeout_ms=15000,
            request_timeout_ms=40000,
            max_poll_interval_ms=300000,
        )
        self.started_at = time.time()
        self._thread = threading.Thread(target=self._run, name="q1c-tail", daemon=True)
        self._thread.start()
        return (f"session {self.session} started — group={self.group_id}, "
                f"committed offsets are read from the broker")

    def stop(self):
        if not (self._thread and self._thread.is_alive()):
            return "listener is not running"
        self._stop.set()
        self._thread.join(timeout=20)
        self._commit_and_read(self.consumer)   # final commit from the main thread
        self.stopped_at = time.time()
        with self.lock:
            self.stop_mark = {
                "at": self.stopped_at,
                "session": self.session,
                "committed": dict(self.committed),
                "last": {p: i["last"] for p, i in self.per_part.items()},
            }
        try:
            self.consumer.close(autocommit=False)
        except TypeError:
            self.consumer.close()
        self.consumer = None
        return f"session {self.session} stopped — consumer closed, offsets committed"

    # ---------------------------------------------------------------- internals
    def _run(self):
        consumer = self.consumer
        while not self._stop.is_set():
            try:
                batches = consumer.poll(timeout_ms=1000, max_records=self.max_records)
            except Exception as exc:            # noqa: BLE001
                self.last_error = f"poll: {type(exc).__name__}: {exc}"
                time.sleep(1.0)
                continue
            if batches:
                for tp, records in batches.items():
                    for rec in records:
                        self._ingest(tp.partition, rec)
                self._commit_and_read(consumer)
            self._refresh_end(consumer)

    def _commit_and_read(self, consumer):
        """commit() returns None in kafka-python 3.x — read the offsets back from the broker."""
        if consumer is None:
            return
        try:
            consumer.commit()                   # commits each partition's current position
        except Exception as exc:                # noqa: BLE001
            self.last_error = f"commit: {type(exc).__name__}: {exc}"
            return
        out = {}
        for tp in consumer.assignment():
            try:
                md = consumer.committed(tp)
            except Exception:                   # noqa: BLE001
                continue
            if md is not None:
                out[tp.partition] = getattr(md, "offset", md)
        if out:
            self.committed = out

    def _refresh_end(self, consumer):
        parts = consumer.assignment()
        if not parts:
            return
        try:
            self.end = {tp.partition: off
                        for tp, off in consumer.end_offsets(list(parts)).items()}
        except Exception as exc:                # noqa: BLE001
            self.last_error = f"end_offsets: {type(exc).__name__}: {exc}"

    def _ingest(self, part, rec):
        try:
            payload = json.loads(rec.value.decode("utf-8"))
            if not isinstance(payload, dict):
                raise ValueError("not an object")
        except Exception:                       # noqa: BLE001
            payload = {}
            with self.lock:
                self.bad_json += 1

        now = time.time()
        with self.lock:
            self.count += 1
            info = self.per_part.setdefault(part, {"first": None, "last": None, "count": 0})
            prev_last = info["last"]
            if info["first"] is None:
                info["first"] = rec.offset    # first record of this partition, ever
            first_here = part not in self.session_first
            if first_here:
                self.session_first[part] = rec.offset
                # a jump at the START of a session is a resume, not a hole: record it
                if (prev_last is not None and rec.offset > prev_last + 1
                        and self.stop_mark is not None):
                    info["resumed_from"] = (prev_last, rec.offset)
            if prev_last is not None:
                if rec.offset <= prev_last:
                    self.dup_offsets += 1       # replayed offset -> at-least-once boundary
                elif rec.offset > prev_last + 1 and not first_here:
                    self.skipped += rec.offset - prev_last - 1
            info["last"] = rec.offset if prev_last is None else max(prev_last, rec.offset)
            info["count"] += 1
            self.recent.append({
                "t": now,
                "part": part,
                "off": rec.offset,
                "key": (rec.key or b"").decode("utf-8", "replace"),
                "wiki": payload.get("wiki"),
                "type": payload.get("type"),
                "title": payload.get("title"),
                "user": payload.get("user"),
                "bot": payload.get("bot"),
                "ts": payload.get("timestamp"),
                "old": (payload.get("length") or {}).get("old"),
                "new": (payload.get("length") or {}).get("new"),
            })
            self.samples.append((now, self.count))
            # business key; `id` is absent on log events (~2.5 % of the stream), so
            # those fall back to a composite that is still stable per event
            if payload.get("id") is not None:
                key = ("id", payload.get("wiki"), payload.get("id"))
            else:
                key = ("log", payload.get("wiki"), payload.get("type"),
                       payload.get("title"), payload.get("timestamp"), payload.get("user"))
            if key in self.seen_ids:
                self.dup_ids += 1               # duplicate business key (wiki, id)
            else:
                self.seen_ids.add(key)
        if payload:
            ts = payload.get("timestamp")
            if isinstance(ts, (int, float)):
                if self.first_ts is None:
                    self.first_ts = ts
                self.last_ts = ts

    # ---------------------------------------------------------------- read-only API
    def snapshot(self):
        with self.lock:
            return {
                "session": self.session, "count": self.count,
                "dup_offsets": self.dup_offsets, "skipped": self.skipped,
                "bad_json": self.bad_json, "dup_ids": self.dup_ids,
                "unique_keys": len(self.seen_ids),
                "per_part": {p: dict(i) for p, i in self.per_part.items()},
                "committed": dict(self.committed), "end": dict(self.end),
                "recent": list(self.recent)[-self.table_rows * 3:],
                "first_ts": self.first_ts, "last_ts": self.last_ts,
                "alive": bool(self._thread and self._thread.is_alive()),
                "last_error": self.last_error,
            }

    def rate(self, window=10.0):
        with self.lock:
            if not self.samples:
                return 0.0
            now, total = self.samples[-1]
            base = total
            for t, c in self.samples:
                if t >= now - window:
                    base = c
                    break
            span = max(now - (now - window), 0.001)
            return (total - base) / span

    def wait_for_first(self, timeout=180.0, poll=1.0):
        t0 = time.time()
        while time.time() - t0 < timeout:
            if self.count > 0:
                return True
            time.sleep(poll)
        return False


In [3]:
_CSS = """
<style>
.q1c { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px; color: #0f172a; }
.q1c h4 { margin: 0 0 6px 0; font-size: 13px; }
.q1c .chips { display: flex; flex-wrap: wrap; gap: 6px; margin: 0 0 8px 0; }
.q1c .chip { background: #f1f5f9; border: 1px solid #e2e8f0; border-radius: 6px; padding: 3px 8px; }
.q1c .chip b { color: #0e7490; }
.q1c table { border-collapse: collapse; margin: 0 0 10px 0; }
.q1c th, .q1c td { border: 1px solid #e2e8f0; padding: 2px 6px; text-align: right; white-space: nowrap; }
.q1c th { background: #f8fafc; text-align: center; font-weight: 600; }
.q1c td.l, .q1c th.l { text-align: left; }
.q1c .lag { color: #b45309; font-weight: 600; }
.q1c .ok { color: #047857; font-weight: 600; }
.q1c .muted { color: #64748b; }
</style>
"""


def _esc(x):
    if x is None:
        return "-"
    return str(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _cut(x, n):
    s = "-" if x is None else str(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def q1c_panel(tail):
    """One HTML block: counters, per-partition offsets/lag, and the message tail."""
    s = tail.snapshot()
    parts = sorted(set(s["per_part"]) | set(range(PARTITIONS)))
    alive = "listening" if s["alive"] else "STOPPED"
    chips = [
        ("state", alive), ("session", s["session"]),
        ("consumed", s["count"]), ("msg/s (10s)", f"{tail.rate():.1f}"),
        ("unique (wiki,id)", s["unique_keys"]), ("dup offsets", s["dup_offsets"]),
        ("dup keys", s["dup_ids"]), ("skipped", s["skipped"]),
        ("event time", f"{_fmt_epoch(s['first_ts'])} → {_fmt_epoch(s['last_ts'])}"),
    ]
    out = [_CSS, '<div class="q1c">']
    out.append(f"<h4>LIVE — topic <b>{_esc(TOPIC)}</b> · group {_esc(GROUP_ID)} · {alive}</h4>")
    out.append('<div class="chips">' + "".join(
        f'<span class="chip">{_esc(k)} <b>{_esc(v)}</b></span>' for k, v in chips) + "</div>")

    out.append('<table><tr><th>part</th><th>first</th><th>last</th><th>records</th>'
               '<th>committed</th><th>end</th><th>lag</th></tr>')
    for p in parts:
        info = s["per_part"].get(p, {})
        com = s["committed"].get(p)
        end = s["end"].get(p)
        lag = (end - com) if (end is not None and com is not None) else None
        cls = ' class="lag"' if (lag or 0) > 0 else ""
        out.append(
            f'<tr><td class="l">P{p}</td><td>{_esc(info.get("first"))}</td>'
            f'<td>{_esc(info.get("last"))}</td><td>{_esc(info.get("count"))}</td>'
            f'<td>{_esc(com)}</td><td>{_esc(end)}</td><td{cls}>{_esc(lag)}</td></tr>')
    out.append("</table>")

    out.append('<table><tr><th class="l">recv</th><th>part</th><th>offset</th>'
               '<th class="l">key</th><th class="l">wiki</th><th class="l">type</th>'
               '<th class="l">title</th><th class="l">user</th><th>bot</th>'
               '<th class="l">event</th><th>Δ bytes</th></tr>')
    for r in s["recent"][-tail.table_rows:][::-1]:
        delta = "-"
        if isinstance(r["new"], int):
            delta = r["new"] - (r["old"] or 0)
        out.append(
            f'<tr><td class="l muted">{_dt.datetime.fromtimestamp(r["t"]).strftime("%H:%M:%S")}</td>'
            f'<td>P{r["part"]}</td><td>{r["off"]}</td><td class="l">{_esc(_cut(r["key"], 18))}</td>'
            f'<td class="l">{_esc(_cut(r["wiki"], 14))}</td><td class="l">{_esc(r["type"])}</td>'
            f'<td class="l">{_esc(_cut(r["title"], 38))}</td>'
            f'<td class="l">{_esc(_cut(r["user"], 20))}</td><td>{_esc(r["bot"])}</td>'
            f'<td class="l">{_fmt_epoch(r["ts"])}</td><td>{_esc(delta)}</td></tr>')
    out.append("</table>")
    if s["last_error"]:
        out.append(f'<div class="muted">last error: {_esc(s["last_error"])}</div>')
    out.append("</div>")
    return "".join(out)


def q1c_show(tail):
    """Render the panel once (in a kernel) or a one-line summary (plain python)."""
    if _in_kernel():
        from IPython.display import display, HTML
        display(HTML(q1c_panel(tail)))
    else:
        s = tail.snapshot()
        print(f"consumed={s['count']}  session={s['session']}  alive={s['alive']}")


def _in_kernel():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:                            # noqa: BLE001
        return False


def q1c_live_view(tail, seconds=VIEW_SECONDS, refresh=1.5):
    """Refresh the panel in place until `seconds` elapse (Ctrl-C stops the view only)."""
    handle = None
    if _in_kernel():
        try:
            from IPython.display import display, HTML
            handle = display(HTML(q1c_panel(tail)), display_id=True)
        except Exception:                        # noqa: BLE001
            handle = None

    if handle is not None and hasattr(handle, "update"):
        def update():
            handle.update(HTML(q1c_panel(tail)))
    else:
        def update():                            # line-per-refresh fallback
            s = tail.snapshot()
            last = s["recent"][-1] if s["recent"] else {}
            print(f"[{_dt.datetime.now():%H:%M:%S}] session={s['session']} "
                  f"consumed={s['count']} rate={tail.rate():.1f}/s alive={s['alive']} "
                  f"lag={sum((s['end'].get(p) or 0) - (s['committed'].get(p) or 0) for p in s['end'])} "
                  f"last=P{last.get('part')}@{last.get('off')} {_cut(last.get('wiki'), 14)}")

    t0 = time.time()
    print(f"refreshing for {seconds}s — Ctrl-C leaves the listener running")
    try:
        while time.time() - t0 < seconds:
            update()
            time.sleep(refresh)
    except KeyboardInterrupt:
        print("view stopped (listener still running)")
    update()


def q1c_topic_lag():
    """End offsets (broker) vs committed offsets (group) — works while nothing consumes."""
    probe = KafkaConsumer(bootstrap_servers=BOOTSTRAP.split(","), client_id="q1c-probe")
    try:
        parts_set = probe.partitions_for_topic(TOPIC)
    except Exception:                            # noqa: BLE001
        parts_set = None
    parts = [TopicPartition(TOPIC, p) for p in
             sorted(parts_set if parts_set else range(PARTITIONS))]
    try:
        end = probe.end_offsets(parts)
    finally:
        probe.close()
    grp = KafkaConsumer(bootstrap_servers=BOOTSTRAP.split(","),
                        group_id=GROUP_ID, client_id="q1c-offsets")
    committed = {}
    try:
        for tp in parts:                        # committed() takes a single partition
            try:
                md = grp.committed(tp)
            except Exception:                  # noqa: BLE001
                md = None
            committed[tp.partition] = (getattr(md, "offset", md)
                                       if md is not None else None)
    finally:
        grp.close()
    rows = []
    for tp in parts:
        com = committed.get(tp.partition)
        e = end.get(tp)
        rows.append({"part": tp.partition, "committed": com, "end": e,
                     "lag": (e - com) if (com is not None and e is not None) else None})
    return rows


def q1c_table(rows, cols):
    """Fixed-width text table (column width follows the widest cell)."""
    widths = {c: max(len(c), 8, *(len(_esc(r.get(c))) for r in rows)) if rows else max(len(c), 8)
              for c in cols}
    head = "  ".join(f"{c:>{widths[c]}}" for c in cols)
    lines = [head, "-" * len(head)]
    for r in rows:
        lines.append("  ".join(f"{_esc(r.get(c)):>{widths[c]}}" for c in cols))
    return chr(10).join(lines)


def q1c_stop_summary(tail):
    m = tail.stop_mark or {}
    print(f"consumer stopped at {_dt.datetime.fromtimestamp(m.get('at', time.time())):%H:%M:%S} "
          f"(session {m.get('session')})")
    print("last processed offset and committed offset per partition:")
    rows = [{"part": p, "last_processed": v, "next_expected": v + 1,
             "committed": m.get("committed", {}).get(p)}
            for p, v in sorted(m.get("last", {}).items())]
    print(q1c_table(rows, ["part", "last_processed", "next_expected", "committed"]))


def q1c_buffer_watch(seconds=STOP_SECONDS, every=5):
    """Show the topic filling up while the consumer is down (producer still running)."""
    print(f"watching the topic for {seconds}s with the consumer DOWN "
          f"(producer must keep running)...")
    t0 = time.time()
    first = None
    while time.time() - t0 <= seconds:
        rows = q1c_topic_lag()
        total_end = sum(r["end"] or 0 for r in rows)
        total_lag = sum(r["lag"] or 0 for r in rows)
        if first is None:
            first = total_end
        print(f"[+{time.time() - t0:5.0f}s] end offsets={total_end}  "
              f"uncommitted lag={total_lag}  (accumulated during downtime={total_end - first})")
        time.sleep(every)


def q1c_resume_report(tail):
    m = tail.stop_mark or {}
    if not m:
        print("no stop recorded — run the stop cell first (this is the Q1c downtime)")
        return
    print(f"session {tail.session} started; consumer resumes from the group's committed offsets")
    t0 = time.time()
    while not tail.session_first and time.time() - t0 < 30:   # wait for the first batch
        time.sleep(0.5)
    if not tail.session_first:
        print("no record consumed yet — the topic is empty; run the producer again")
        return
    rows = []
    for p in sorted(tail.session_first):
        rows.append({
            "part": p,
            "committed_at_stop": m["committed"].get(p),
            "last_processed": m["last"].get(p),
            "first_after_resume": tail.session_first[p],
        })
    print(q1c_table(rows, ["part", "committed_at_stop", "last_processed", "first_after_resume"]))


def q1c_verify(tail):
    """Q1c verdict: lost? duplicated? resumes from the last committed offset?"""
    m = tail.stop_mark or {}
    if not m:
        print("run the stop/resume sequence first")
        return
    if tail.session < 2:
        print("restart the listener (session 2) before verifying")
        return
    rows, lost, dup_offsets = [], 0, 0
    for p in sorted(set(m["last"]) | set(tail.session_first)):
        last_before = m["last"].get(p)
        committed = m["committed"].get(p)
        first_after = tail.session_first.get(p)
        expected = (last_before + 1) if last_before is not None else None
        gap = None
        if expected is not None and first_after is not None:
            gap = first_after - expected          # >0 => messages lost, <0 => replayed
            lost += max(gap, 0)
            dup_offsets += max(-gap, 0)
        rows.append({"part": p, "last_before_stop": last_before,
                     "committed": committed, "expected_next": expected,
                     "first_after_resume": first_after, "offset_gap": gap})
    print("resume check (per partition):")
    print(q1c_table(rows, ["part", "last_before_stop", "committed", "expected_next",
                           "first_after_resume", "offset_gap"]))
    s = tail.snapshot()
    print()
    print(f"records consumed in total       : {s['count']}")
    print(f"distinct business keys          : {s['unique_keys']}")
    print(f"business keys seen twice        : {s['dup_ids']}  (source-side echo or id-less "
          f"log events — not caused by the restart)")
    print(f"replayed offsets (counter)      : {s['dup_offsets']}  (cross-check: {dup_offsets})")
    print(f"offsets skipped inside a session: {s['skipped']}")
    print(f"records lost across the restart : {lost}")
    print()
    ok = (lost == 0 and s["dup_offsets"] == 0)
    print("VERDICT: " + ("no records lost and no offset replayed → the consumer resumed at "
                         "the last committed offset (offset_gap = 0 on every partition)"
                         if ok else
                         "loss or replayed offsets detected — read the table above"))
    if ok and s["dup_ids"]:
        print(f"NOTE: the {s['dup_ids']} repeated business keys are duplicates that already "
              "exist in the topic (the SSE feed echoes some events across eqiad/codfw, and "
              "log events carry no id). This is why processing dedupes on (wiki, id) with a "
              "fallback key instead of trusting every record.")
    elif ok:
        print("NOTE: no duplicate business key either — nothing to dedupe in this window.")


def q1c_post_run(tail, seconds=30):
    """After the producer script has exited: the listener keeps its position."""
    a = q1c_topic_lag()
    before = sum(r["end"] or 0 for r in a)
    t_before = _fmt_epoch(tail.last_ts)
    print(f"end offsets now                 : {before}")
    print(f"newest event time consumed      : {t_before}")
    print(f"waiting {seconds}s for new data ...")
    time.sleep(seconds)
    b = q1c_topic_lag()
    after = sum(r["end"] or 0 for r in b)
    print(f"end offsets after {seconds}s      : {after}  (delta={after - before})")
    print(f"listener consumed               : {tail.count} records, "
          f"lag={sum(r['lag'] or 0 for r in b)}")
    if after - before == 0:
        print("VERDICT: the producer has stopped — the topic is idle and the listener "
              "is fully caught up (no data arrives behind it)")
    else:
        print(f"VERDICT: {after - before} new records arrived; the producer is still running")


## Step 1 — start listening (before the producer)

Creates the consumer on its own group `wiki-monitor-q1c` and starts the background
thread. Nothing is printed in the panel until the producer writes, which is exactly
the point: the listener is already positioned on every partition.


In [4]:
tail = KafkaTail(BOOTSTRAP, TOPIC, GROUP_ID, table_rows=TABLE_ROWS, max_records=MAX_RECORDS)
print(tail.start())
print("waiting for the first record (start the producer in a terminal now)...")
print("first record received:", tail.wait_for_first(timeout=180))
q1c_show(tail)

session 1 started — group=wiki-monitor-q1c, committed offsets are read from the broker
waiting for the first record (start the producer in a terminal now)...
first record received: True


## Step 2 — live panel: the topic as it arrives

Counters, per-partition `first/last/committed/end/lag`, and the newest messages with
their key, wiki, title, editor and byte delta. `lag` is `end − committed`, i.e. how
far the consumer is behind the broker right now.


In [5]:
q1c_live_view(tail, seconds=VIEW_SECONDS)

refreshing for 120s — Ctrl-C leaves the listener running


## Step 3 — Q1c: stop the consumer, keep the producer running

`stop()` closes the consumer after a final commit, so the offsets that survive are
exactly the records the panel showed. Then watch the topic accumulate while nothing
consumes it.


In [6]:
print(tail.stop())
q1c_stop_summary(tail)

session 1 stopped — consumer closed, offsets committed
consumer stopped at 13:40:08 (session 1)
last processed offset and committed offset per partition:
    part  last_processed  next_expected  committed
--------------------------------------------------
       0            1456           1457       1457
       1            1363           1364       1364
       2            2779           2780       2780
       3           10136          10137      10137
       4            1657           1658       1658
       5           19087          19088      19088


In [7]:
q1c_buffer_watch(STOP_SECONDS, every=5)

watching the topic for 120s with the consumer DOWN (producer must keep running)...
[+    0s] end offsets=36738  uncommitted lag=254  (accumulated during downtime=0)
[+    5s] end offsets=36973  uncommitted lag=489  (accumulated during downtime=235)
[+   10s] end offsets=37195  uncommitted lag=711  (accumulated during downtime=457)
[+   16s] end offsets=37394  uncommitted lag=910  (accumulated during downtime=656)
[+   21s] end offsets=37661  uncommitted lag=1177  (accumulated during downtime=923)
[+   26s] end offsets=37889  uncommitted lag=1405  (accumulated during downtime=1151)
[+   31s] end offsets=38153  uncommitted lag=1669  (accumulated during downtime=1415)
[+   36s] end offsets=38389  uncommitted lag=1905  (accumulated during downtime=1651)
[+   41s] end offsets=38664  uncommitted lag=2180  (accumulated during downtime=1926)
[+   46s] end offsets=38877  uncommitted lag=2393  (accumulated during downtime=2139)
[+   51s] end offsets=39065  uncommitted lag=2581  (accumulated duri

## Step 4 — restart: resume from the last committed offset

Same group id, same topic. Kafka hands back the partition's committed position, so the
first record after the restart continues the numbering — nothing before it is replayed.


Nothing before it is replayed and nothing between committed and end is skipped.


In [8]:
print(tail.start())
q1c_resume_report(tail)

session 2 started — group=wiki-monitor-q1c, committed offsets are read from the broker
session 2 started; consumer resumes from the group's committed offsets
    part  committed_at_stop  last_processed  first_after_resume
---------------------------------------------------------------
       0               1457            1456                1457
       1               1364            1363                1364
       2               2780            2779                2780
       3              10137           10136               10137
       4               1658            1657                1658
       5              19088           19087               19088


In [9]:
q1c_live_view(tail, seconds=60)

refreshing for 60s — Ctrl-C leaves the listener running


## Step 5 — verification: lost? duplicated?

`offset_gap = first_after_resume − (last_before_stop + 1)` per partition.
`0` everywhere = the restart lost nothing; a negative value would be a replay
(duplicate), a positive value a hole (lost records). The business-key column checks the
same thing independently via `(wiki, id)`.


In [10]:
q1c_verify(tail)

resume check (per partition):
    part  last_before_stop  committed  expected_next  first_after_resume  offset_gap
------------------------------------------------------------------------------------
       0              1456       1457           1457                1457           0
       1              1363       1364           1364                1364           0
       2              2779       2780           2780                2780           0
       3             10136      10137          10137               10137           0
       4              1657       1658           1658                1658           0
       5             19087      19088          19088               19088           0

records consumed in total       : 48794
distinct business keys          : 48652
business keys seen twice        : 142  (source-side echo or id-less log events — not caused by the restart)
replayed offsets (counter)      : 0  (cross-check: 0)
offsets skipped inside a session: 0
records los

## Step 6 — after the producer script ends

The listener stays where it is; the topic becomes idle. Run this once the producer
terminal prints its 12-minute summary: end offsets must not move any more.


In [11]:
q1c_post_run(tail, seconds=30)

end offsets now                 : 49024
newest event time consumed      : 13:45:12
waiting 30s for new data ...
end offsets after 30s      : 50461  (delta=1437)
listener consumed               : 50461 records, lag=0
VERDICT: 1437 new records arrived; the producer is still running


In [12]:
print(tail.stop())
print("listener closed; offsets are committed and survive the kernel")

session 2 stopped — consumer closed, offsets committed
listener closed; offsets are committed and survive the kernel


## Notes for the report

* **`id` is not always present.** `type=log` events (≈2.5 % of the stream) carry
  `log_id`/`log_type` instead of `id`, so a `(wiki, id)` key would collapse all of them
  into one key and report false duplicates. The listener therefore keys on
  `(wiki, id)` when `id` exists and on `(wiki, type, title, timestamp, user)` otherwise.
* **At-least-once, not exactly-once.** `poll()` → process → `commit()`. A crash between
  the two replays the last batch. `commit()` after each batch keeps the replay window
  to one batch; deduplicating on `(wiki, id)` makes the replay harmless, which is why
  the producer keys on `wiki` while the *consumer* dedupes on the pair.
* **Why manual commit.** With `enable_auto_commit=True` the offsets are committed on a
  timer, so a stop can silently commit records that were never processed (loss) or
  reprocess committed ones (duplicates). `auto_offset_reset="earliest"` only applies to
  a group with no committed offset at all — after the first run the group's own offsets
  win, which is the behaviour being demonstrated.
* **Lag is the decoupling metric.** During the 2-minute downtime the sum of `end −
  committed` grows by roughly `rate × 120 s`; that number is the buffer proving the
  producer does not depend on the consumer.
* **Fresh demo run.** To start from offset 0 with this group again, stop the listener
  and delete the group's offsets:
  `podman exec skc-kafka /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:29092 --delete --group wiki-monitor-q1c`
* One `KafkaTail` per kernel: re-running **Step 1** after **Step 4** would create a
  second consumer in the same group and silently steal partitions. Run the cells in
  order; `tail.stop()` first if you want to start over.
